# 010. Wstęp, NumPy i Nowoczesny Ekosystem Pandas

Ten notebook demonstruje fundamenty obliczeniowe biblioteki Pandas: wektoryzację NumPy, zaawansowane instrukcje warunkowe (`np.where`, `np.select`), mechanizm **Copy-on-Write** oraz diagnostykę tabeli.


## 1. Wektoryzacja vs pętle Pythona (`np.where`)


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Generowanie próbki danych (100 000 wierszy)
np.random.seed(42)
n = 100_000
df = pd.DataFrame({
    'Klient_ID': np.arange(1, n + 1),
    'Kwota': np.random.uniform(50, 5000, size=n),
    'Wiek': np.random.randint(18, 75, size=n)
})

# Wektorowy warunek zamiast pętli (100x szybciej!)
df['Segment_Wartosci'] = np.where(df['Kwota'] > 2500, 'Wysoka Wartość', 'Standard')
df.head()


## 2. Złożone warunki logiczne: `np.select()`


In [ ]:
# Definiujemy listę warunków logicznych
warunki = [
    (df['Wiek'] < 26) & (df['Kwota'] > 2000),
    (df['Wiek'] >= 26) & (df['Kwota'] > 3500),
    (df['Kwota'] < 500)
]

kategorie = [
    'Młody VIP',
    'Doświadczony VIP',
    'Niski Koszyk'
]

df['Kategoria_Klienta'] = np.select(warunki, kategorie, default='Standard')
df['Kategoria_Klienta'].value_counts()


## 3. Nowoczesny Pandas: Copy-on-Write (CoW) w akcji


In [ ]:
# Włączenie trybu Copy-on-Write
pd.set_option('mode.copy_on_write', True)

df_baza = pd.DataFrame({'Miasto': ['Warszawa', 'Kraków', 'Gdańsk'], 'Wartosc': [100, 200, 300]})

# Tworzymy wycinek:
wycinek = df_baza[df_baza['Wartosc'] > 150]

# W starym Pandas to rzucało SettingWithCopyWarning.
# W trybie CoW zmiana w wycinku tworzy automatyczną kopię i NIE narusza oryginału!
wycinek['Nowa_Kolumna'] = 'Promocja'

print("Wycinek ze zmianą:")
print(wycinek)
print("\nOryginał pozostał nienaruszony:")
print(df_baza)


## 4. Diagnostyka struktury danych i zużycia pamięci RAM


In [ ]:
print("Kształt tabeli (wiersze, kolumny):", df.shape)
print("\nTypy danych per kolumna:")
print(df.dtypes)
print("\nPełny raport diagnostyczny pamięci:")
df.info(memory_usage='deep')
